# Ordenamiento Territorial — Análisis de Conflictos de Uso del Suelo

**Bloque:** A — Gestión | **Línea temática:** `ordenamiento_territorial` | **Variable principal:** porcentaje de suelo con conflicto de uso (%)

---

## ¿Qué es el Ordenamiento Territorial en Colombia?

El Ordenamiento Territorial (OT) es el proceso de planificación que busca organizar el uso y ocupación del suelo de manera armónica con la conservación de la biodiversidad, la gestión del riesgo y el desarrollo socioeconómico. El instrumento central es el **Plan de Ordenamiento Territorial (POT)** — o PBOT para municipios medianos y EOT para municipios pequeños — regulado por la **Ley 388 de 1997 (Ley de Desarrollo Territorial)**.

Un **conflicto de uso del suelo** ocurre cuando la actividad que se realiza sobre un predio (uso actual) no coincide con la vocación natural del suelo según sus características físicas, ecológicas y normativas (uso potencial). Este desajuste produce pérdida de ecosistemas, degradación de suelos, reducción de la oferta hídrica y aumento de la vulnerabilidad ante desastres.

## Marco normativo clave

| Norma | Contenido relevante |
|---|---|
| **Ley 388/1997** | POT/PBOT/EOT: clasificación del suelo en urbano, rural y de expansión |
| **Ley 99/1993** | SINA; determinantes ambientales de superior jerarquía para los POT |
| **Decreto 1076/2015** | Reglamentación ambiental única; rondas hídricas, POMCA como determinantes |
| **Ley 1523/2012** | Riesgo de desastres como condicionante territorial obligatorio |
| **Resolución 0058/2025 (Minvivienda)** | Modelo LADM_COL-POT para interoperabilidad de datos cartográficos |

## Instituciones clave

| Institución | Rol en el OT |
|---|---|
| **MADS** | Determinantes ambientales nacionales |
| **MVCT** | Lineamientos técnicos para POT, vivienda y ciudad |
| **DNP** | Articulación entre planeación territorial y financiamiento |
| **IGAC** | Cartografía base, catastro multipropósito, LADM_COL |
| **IDEAM** | Corine Land Cover, cambios de cobertura, clima |
| **CARs** | POMCA como determinante ambiental de los POT municipales |
| **Municipios** | Formulación, adopción y actualización del POT |

## Preguntas que responde este notebook

1. ¿Qué porcentaje del territorio municipal tiene conflicto de uso del suelo y cómo ha evolucionado en el tiempo?
2. ¿Existe una tendencia estadísticamente significativa en el incremento del conflicto de uso antes y después del Acuerdo de Paz (2016)?
3. ¿El Acuerdo de Paz de 2016 marcó un punto de quiebre en la dinámica de uso del suelo? (análisis pre-2016 vs post-2016)
4. ¿Qué proyección a 5–10 años arroja el modelo seleccionado para la expansión del conflicto de uso?

> **Ficha de dominio completa:** [`docs/fuentes/ordenamiento_territorial.md`](../../docs/fuentes/ordenamiento_territorial.md)  
> **Decisiones metodológicas:** [`docs/decisiones.md`](../../docs/decisiones.md)

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "ordenamiento_territorial"
VARIABLE = "superficie_km2"
UNIDAD = "km²"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `ordenamiento_territorial` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "ordenamiento_territorial.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

### Fuentes de datos para conflicto de uso del suelo en Colombia

El conflicto de uso del suelo es una variable derivada que se calcula cruzando el **mapa de uso actual** con el **mapa de uso potencial** del suelo. Las fuentes principales son:

| Fuente | Producto | Escala | Acceso |
|---|---|---|---|
| **IGAC + IDEAM** | Mapa de Conflictos de Uso del Suelo de Colombia | 1:100.000 | [igac.gov.co](https://www.igac.gov.co) |
| **IDEAM (Corine Land Cover)** | Coberturas de la tierra en Colombia | 1:100.000 | [ideam.gov.co](http://www.ideam.gov.co) |
| **IGAC** | Catastro rural y catastro multipropósito | Predial | [igac.gov.co](https://www.igac.gov.co) |
| **TerriData (DNP)** | Indicadores territoriales municipales (series anuales) | Municipal | [terridata.dnp.gov.co](https://terridata.dnp.gov.co) |

**Nota sobre la variable:** El **porcentaje de suelo con conflicto de uso (%)** mide la proporción del área municipal donde el uso actual difiere del potencial según la vocación del suelo. Valores altos (>40%) indican presión severa sobre ecosistemas y riesgo de degradación ambiental.

**Frecuencia:** Los mapas de conflictos se actualizan de forma multitemporal (cada 5–10 años). Para análisis de tendencia, se usan los cortes disponibles de Corine Land Cover (1987–2000, 2000–2002, 2005–2009, 2010–2012, 2014, 2018, 2022).

> Colocar el archivo en `data/raw/` y ajustar la ruta. Serie típica: anual por municipio, ~20–30 observaciones.

In [ ]:
# df = load_csv("data/raw/ordenamiento_territorial.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "superficie_km2": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 1b. Estructura Ecologica Principal (EEP) e indicadores de presion territorial

La **EEP** es el conjunto de areas y ecosistemas estrategicos que sostienen los procesos ecologicos del territorio. Es la base para la zonificacion ambiental en el POT (Ley 388/1997).

| Componente EEP | Norma | Funcion |
|---|---|---|
| Paramos | Ley 1930/2018 | Regulacion hidrica, provision agua |
| Humedales | Ley 357/1997 (Ramsar) | Regulacion hidrica, biodiversidad |
| Rondas hidricas | Res. 957/2018 | Proteccion cauces, zonas de inundacion |
| Areas Protegidas | Decreto 1076/2015 | Conservacion biodiversidad |

**Indicadores de presion:**
- **TCCN (Tasa de Cambio de Coberturas Naturales):** perdida anual en % de ecosistemas estrategicos
- **IACAL (Indice de Alteracion Potencial de la Calidad del Agua):** presion por vertimientos
- **IACAL formula:** Cargas_contaminantes_vertidas / Oferta_hidrica_disponible

In [ ]:
# EEP sintetica + TCCN + IACAL por municipio
np.random.seed(55)
n = len(df)

# Coberturas EEP (ha) -- tendencia decreciente por expansion urbana/agricola
componentes_eep = ['Paramo', 'Humedal', 'Bosque_Andino', 'Ronda_Hidrica']
n_muni = 12
municipios = [f'Mpio-{i+1:02d}' for i in range(n_muni)]
ha_eep = {
    comp: np.random.uniform(100, 5000, n_muni)
    for comp in componentes_eep
}
df_eep = pd.DataFrame({'municipio': municipios, **ha_eep})
df_eep['total_eep_ha'] = df_eep[componentes_eep].sum(axis=1)

# TCCN: tasa de cambio de coberturas naturales (%/ano)
df_eep['tccn_pct'] = np.random.uniform(-3.5, -0.2, n_muni).round(2)  # negativo = perdida

# IACAL: cargas vertimientos / oferta hidrica (0-1 bajo; >2 critico)
carga_dbo_ton = np.random.uniform(50, 800, n_muni)  # ton DBO/ano
oferta_hm3 = np.random.uniform(5, 200, n_muni)     # hm3/ano
df_eep['iacal'] = (carga_dbo_ton / oferta_hm3).round(3)

def iacal_cat(v):
    if v < 0.1: return 'Muy bajo'
    if v < 1.0: return 'Bajo'
    if v < 2.0: return 'Medio'
    if v < 5.0: return 'Alto'
    return 'Muy alto'

df_eep['iacal_cat'] = df_eep['iacal'].apply(iacal_cat)

from estadistica_ambiental.config import IUA_THRESHOLDS
print('IUA_THRESHOLDS (referencia para IACAL):', IUA_THRESHOLDS)
print(f'\nIACAL | min={df_eep["iacal"].min():.3f} max={df_eep["iacal"].max():.3f}')
print(f'Municipios IACAL Alto/Muy Alto: {(df_eep["iacal"] >= 2).sum()}/{n_muni}')
print(f'TCCN promedio: {df_eep["tccn_pct"].mean():.2f}%/ano (negativo = perdida coberturas)')
df_eep[['municipio','total_eep_ha','tccn_pct','iacal','iacal_cat']]

## 2. Validación y EDA

### Consideraciones de validación para conflicto de uso del suelo

La variable de conflicto de uso es un porcentaje (0–100%). La función `validate()` aplicará los siguientes controles:

- **Rango físico:** 0% ≤ conflicto ≤ 100%.
- **Consistencia temporal:** Detectar saltos abruptos de >15 puntos porcentuales entre períodos consecutivos (pueden indicar cambio de metodología cartográfica, no cambio real del territorio).
- **Desactualización:** A 2023, cerca del 80–87% de los municipios colombianos tenían vigencia de POT vencida. Esto puede introducir sesgo en la variable si el conflicto se deriva del POT vigente versus el uso real del suelo.

> **ADR-002:** Los picos en el conflicto de uso no son artefactos a eliminar. Pueden reflejar eventos reales como colonización de zonas de páramo, expansión de palma de aceite o coca, y son la señal de interés para el análisis.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_ordenamiento_territorial.html",
       title="EDA — Ordenamiento Territorial", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

### Qué buscar en el análisis visual del conflicto de uso

La serie anual de conflicto de uso del suelo revela dinámicas territoriales de largo plazo vinculadas a procesos socioeconómicos y políticos. Al visualizar, prestar atención a:

- **Tendencia 1990–2015:** Período de conflicto armado. La presencia de grupos armados inhibió la colonización en algunas zonas, "conservando" ecosistemas de facto. El conflicto de uso puede haberse mantenido artificialmente bajo en zonas de alta amenaza de seguridad.
- **Ruptura 2016 (Acuerdo de Paz):** La desmovilización de las FARC abrió zonas antes inaccesibles a la colonización agropecuaria. Estudios del IDEAM muestran un aumento sostenido de la deforestación en el Amazonas colombiano a partir de 2016. El análisis debe separar explícitamente los dos subperíodos.
- **Comparación pre-2016 vs post-2016:** Graficando las dos ventanas temporales por separado, se puede visualizar si el Acuerdo de Paz alteró la trayectoria del conflicto de uso en la región analizada.
- **Estacionalidad:** Para datos anuales, no hay estacionalidad. `plot_seasonal_means` con `period="year"` sirve para visualizar la evolución interanual.

In [ ]:
plot_series(df, "fecha", "superficie_km2", title="Ordenamiento Territorial — superficie_km2 (km²)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "superficie_km2", period="month")
plt.show()

## 3c. IACAL y Estructura Ecologica Principal — Dashboard territorial

La econometria espacial con **modelos SARAR** (Spatial AutoRegressive with AutoRegressive disturbances) permite modelar como la presion territorial en un municipio afecta a sus vecinos:

```
y = rho * W * y + X * beta + u      (SAR: autocorrelacion en y)
u = lambda * W * u + e              (AR: autocorrelacion en errores)
```

Donde W = matriz de contigüedad espacial (reina/torre). Implementacion en Python: `spreg.GM_Combo` de `pysal/spreg`.

> Un IACAL alto en un municipio genera externalidades hidricas negativas aguas abajo (efecto autocorrelacion espacial positiva). Esto justifica el enfoque cuencal del POMCA.

In [ ]:
IACAL_COLORS = {'Muy bajo':'#27ae60','Bajo':'#82e0aa','Medio':'#f1c40f',
                'Alto':'#e67e22','Muy alto':'#e74c3c'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: IACAL por municipio
ax = axes[0]
bar_colors = df_eep['iacal_cat'].map(IACAL_COLORS)
ax.barh(df_eep['municipio'], df_eep['iacal'], color=bar_colors, alpha=0.85)
ax.axvline(2.0, color='red', ls='--', lw=1.5, label='Umbral alto IACAL=2')
ax.set_title('IACAL — Alteracion Potencial Calidad Agua', fontweight='bold')
ax.set_xlabel('IACAL (carga/oferta)'); ax.legend(fontsize=7); ax.grid(axis='x', alpha=0.3)

# Panel B: Composicion EEP por municipio
ax = axes[1]
bottom = np.zeros(n_muni)
colors_eep = ['#1a73e8','#27ae60','#82e0aa','#f1c40f']
for comp, color in zip(componentes_eep, colors_eep):
    ax.bar(range(n_muni), df_eep[comp], bottom=bottom, color=color, alpha=0.85, label=comp)
    bottom += df_eep[comp].values
ax.set_xticks(range(n_muni)); ax.set_xticklabels(df_eep['municipio'], rotation=45, ha='right', fontsize=7)
ax.set_title('Composicion EEP por Municipio (ha)', fontweight='bold')
ax.set_ylabel('Hectareas'); ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)

# Panel C: TCCN vs IACAL
ax = axes[2]
sc = ax.scatter(df_eep['tccn_pct'], df_eep['iacal'],
                c=df_eep['iacal_cat'].map(IACAL_COLORS), s=80, alpha=0.8, edgecolors='gray')
ax.axhline(2.0, color='red', ls='--', lw=1)
ax.axvline(-2.0, color='orange', ls='--', lw=1)
ax.set_xlabel('TCCN (%/ano)'); ax.set_ylabel('IACAL')
ax.set_title('Presion territorial: TCCN vs IACAL\n(SARAR: efecto espacial sobre cuenca)', fontweight='bold')
for _, r in df_eep.iterrows():
    ax.annotate(r['municipio'], (r['tccn_pct'], r['iacal']), fontsize=6)
ax.grid(alpha=0.3)

plt.suptitle('EEP + IACAL + TCCN — Determinantes ambientales POT (Ley 388/1997)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

print('Modelo SARAR (spreg.GM_Combo) requiere: pip install spreg')
print(f'Municipios EEP < 500 ha total: {(df_eep["total_eep_ha"] < 500).sum()} — prioridad restauracion')

## 4. Estadística descriptiva

### Indicadores clave para conflicto de uso del suelo

| Estadístico | Interpretación en OT |
|---|---|
| **Media** | Nivel promedio de conflicto de uso en el período; comparar con medias nacionales del IGAC |
| **Tasa de cambio anual** | ¿Cuántos puntos porcentuales crece el conflicto por año? |
| **Máximo** | ¿Cuál fue el año de mayor presión sobre el suelo? |
| **Diferencia pre/post 2016** | Cambio en la media entre los dos subperíodos |

**Valores de referencia nacionales (IGAC 2018):**
- Colombia: ~53% del territorio tiene algún nivel de conflicto de uso.
- Conflicto por sobreutilización (uso más intensivo que la vocación): ~18% del territorio.
- Conflicto por subutilización (uso menos intensivo que la vocación): ~35% del territorio.
- Sin conflicto (uso adecuado): ~47% del territorio.

**Nota:** El análisis estadístico de la tendencia es más relevante que los valores absolutos, ya que estos dependen de la escala y metodología cartográfica utilizada.

In [ ]:
summarize(df)

## 5. Inferencial

### Estacionariedad y Mann-Kendall por subperíodos: el análisis central de OT

Para datos anuales de conflicto de uso del suelo, el análisis inferencial tiene particularidades importantes:

**Sobre estacionariedad:**
- Con series cortas (<40 puntos anuales), las pruebas ADF y KPSS tienen baja potencia estadística.
- El resultado debe interpretarse con cautela — la no estacionariedad puede reflejar la tendencia real de expansión del conflicto, no un artefacto estadístico.
- Para OT, **no es necesario forzar estacionariedad** para modelar con Random Forest o XGBoost (modelos sin supuesto de estacionariedad).

**Mann-Kendall obligatorio por subperíodo (ADR aplicado a OT):**

El test de Mann-Kendall es el análisis más relevante para esta línea. Se aplica en **dos ventanas temporales separadas** para detectar si el Acuerdo de Paz de 2016 cambió la dirección o intensidad de la tendencia:

| Subperíodo | Hipótesis esperada |
|---|---|
| **Pre-2016 (→2015)** | Tendencia creciente moderada o estable (inhibición por conflicto armado en zonas críticas) |
| **Post-2016 (2016→)** | Tendencia creciente acelerada (colonización de zonas previamente inaccesibles) |

Una diferencia estadísticamente significativa entre los slopes de Sen de los dos subperíodos es evidencia sólida del efecto territorial del Acuerdo de Paz.

> **Nota importante:** No usar `walk_forward` para esta línea. La serie anual es típicamente corta (<40 puntos) y no tiene suficientes observaciones para validación cruzada con ventanas rodantes. Preferir proyección con regresión de tendencia o modelos de suavizamiento.

In [ ]:
ts = df.set_index("fecha")["superficie_km2"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas

### Conflicto de uso del suelo: umbrales de preocupación ambiental

A diferencia de parámetros fisicoquímicos, el conflicto de uso del suelo no tiene umbrales legales fijos en Colombia. Sin embargo, la literatura y los instrumentos de planificación usan las siguientes referencias:

| Nivel de conflicto | % del territorio | Acción planificadora recomendada |
|---|---|---|
| **Bajo** | < 20% | Monitoreo ordinario, incluir en POT |
| **Moderado** | 20–40% | Señal de alerta; actualizar zonificación |
| **Alto** | 40–60% | Proceso urgente de reconversión productiva |
| **Crítico** | > 60% | Intervención inmediata; declaración de áreas protegidas, POMCA |

Estas categorías son de uso cualitativo para la toma de decisiones en los POT y los POMCA. El análisis de excedencias sobre el umbral del 40% puede usarse para determinar cuántos períodos el municipio ha estado en condición de alerta.

**Indicadores complementarios (IGAC):**
- **TCCN (Tasa de Cambio de Coberturas Naturales):** Complementa el conflicto de uso midiendo la velocidad de pérdida de coberturas naturales.
- **IAR (Índice de Afectación del Riesgo):** Relaciona el conflicto de uso con la amenaza por inundaciones y deslizamientos.

In [ ]:
rep = exceedance_report(df["superficie_km2"], variable="superficie_km2")
if rep.empty:
    print("Sin normas colombianas registradas para 'superficie_km2'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

### Tratamiento de datos en series cortas de OT

Las series anuales de conflicto de uso del suelo suelen ser cortas (15–30 observaciones) y con frecuencia tienen valores faltantes en años donde no se realizó levantamiento cartográfico. El preprocesamiento debe ser conservador:

- **Faltantes por ausencia de mapeo:** Interpolación lineal entre los valores disponibles. Dado que el conflicto de uso cambia lentamente (años), la interpolación lineal es físicamente razonable.
- **Faltantes al inicio o final de la serie:** No extrapolar. Recortar la serie al rango con datos disponibles.
- **Cambios de metodología cartográfica:** Si entre dos períodos cambió la escala del mapa o la definición de categorías, documentar el salto y considerarlo como una discontinuidad metodológica, no un cambio real.

> **ADR-002:** No eliminar años con conflicto alto. Un año con >60% de conflicto puede ser el resultado de un evento real (expansión de coca, incendios, megaproyectos) y es la observación más informativa para la gestión territorial.

In [ ]:
df_clean = impute(df.copy(), cols=["superficie_km2"], method="linear")
print(f"Faltantes antes: {df["superficie_km2"].isna().sum()} | "
      f"después: {df_clean["superficie_km2"].isna().sum()}")

## 7. Modelos predictivos

### Limitaciones del modelado predictivo para series cortas de OT

Con series anuales de <40 puntos, las posibilidades de modelado predictivo son limitadas. Las opciones válidas son:

| Modelo | Restricciones | Cuándo usar |
|---|---|---|
| **Random Forest** | Requiere al menos 20–30 puntos para lags significativos | Serie ≥ 20 años; usar lags cortos [1, 2, 3] |
| **XGBoost** | Mismas restricciones que RF; riesgo de sobreajuste en series cortas | Igual que RF; validar con leave-one-out |
| **Regresión de tendencia lineal/polinomial** | Simple pero interpretable; adecuado para proyecciones a 5–10 años | Serie corta, objetivo de proyección de largo plazo |

> **Restricción metodológica importante:** **No usar `walk_forward`** con series cortas anuales. Con n<40, los splits de validación cruzada dejan ventanas de entrenamiento insuficientes para que los modelos aprendan. Usar en cambio validación hold-out simple (últimos 3–5 años como conjunto de prueba).

**Alternativa recomendada:** La proyección de tendencia con regresión lineal segmentada (pre/post 2016) es más honesta estadísticamente que un modelo de machine learning sobreajustado a 20–30 puntos. Permite proyectar escenarios con incertidumbre explícita (intervalo de confianza del 95%).

In [ ]:
ts = df_clean.set_index("fecha")["superficie_km2"]

models = {
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones

### Plantilla de conclusiones (completar con datos reales)

**Estado del conflicto de uso:**
- Media histórica: `X%` del territorio (período `YYYY–YYYY`).
- Comparación con media nacional IGAC 2018: `por encima / por debajo / similar` al promedio colombiano de 53%.

**Análisis por subperíodos (Acuerdo de Paz como ruptura):**
- Mann-Kendall pre-2016: `[creciente / decreciente / sin tendencia]`, slope Sen = `X` pp/año, p = `Y`.
- Mann-Kendall post-2016: `[creciente / decreciente / sin tendencia]`, slope Sen = `X` pp/año, p = `Y`.
- Conclusión: ¿El Acuerdo de Paz aceleró / moderó / no cambió la dinámica del conflicto de uso en la región?

**Excedencias:**
- Número de años con conflicto > 40% (umbral de alerta): `N` de `T` años totales.

**Proyección:**
- Al ritmo actual (slope post-2016), el conflicto de uso alcanzará `X%` en el año `YYYY`.
- Implicación para la actualización del POT municipal.

### Normativa y referencias

- **Ley 388/1997:** POT obligatorio; clasificación del suelo.
- **IGAC (2018):** Mapa de Conflictos de Uso del Suelo de Colombia — referencia para calibrar resultados.
- **IDEAM (Corine Land Cover 2018–2022):** Fuente principal de coberturas actualizadas.
- Registrar decisiones metodológicas en [`docs/decisiones.md`](../../docs/decisiones.md).

## 9. Cómo adaptar a datos reales

### Paso 1: Obtener datos de conflicto de uso del suelo

```python
# Opción A: TerriData (DNP) — series anuales por municipio
# Descargar en: https://terridata.dnp.gov.co/
# Indicador: "Porcentaje de suelo con conflicto de uso" o "Cobertura natural restante"
df = load_csv("data/raw/ot_conflicto_uso_municipio.csv", date_col="año")

# Opción B: Calcular desde Corine Land Cover (requiere SIG)
# Cruzar capa de "Uso actual del suelo" con "Uso potencial del suelo" del IGAC
# Exportar como tabla con columna "conflicto_pct" por año

# Opción C: Usar el Mapa de Conflictos IGAC 2018 como punto de referencia único
```

### Paso 2: Ajustar la variable

Cambiar en la celda de Setup:
```python
VARIABLE = "conflicto_pct"   # Porcentaje de suelo con conflicto de uso
UNIDAD = "%"
```

### Paso 3: Ejecutar Mann-Kendall por subperíodo

```python
from estadistica_ambiental.inference.trend import mann_kendall

# Subperíodo pre-Acuerdo de Paz
ts_pre = df[df["fecha"] < "2016-01-01"].set_index("fecha")["conflicto_pct"]
mk_pre = mann_kendall(ts_pre)
print(f"Pre-2016: {mk_pre['trend']} | slope={mk_pre['slope']:.4f} pp/año | p={mk_pre['pval']:.4f}")

# Subperíodo post-Acuerdo de Paz
ts_post = df[df["fecha"] >= "2016-01-01"].set_index("fecha")["conflicto_pct"]
mk_post = mann_kendall(ts_post)
print(f"Post-2016: {mk_post['trend']} | slope={mk_post['slope']:.4f} pp/año | p={mk_post['pval']:.4f}")
```

### Paso 4: Análisis de excedencias por umbral de alerta

```python
from estadistica_ambiental.inference.intervals import exceedance_report
# Umbral 40%: nivel de alerta en el que la planificación territorial recomienda intervención
rep = exceedance_report(df["conflicto_pct"], variable="conflicto_pct", threshold=40)
print(f"Años en nivel de alerta (>40%): {rep['n_exceedances'].sum()} / {len(df)}")
```

### Paso 5: Proyección lineal segmentada (alternativa a walk_forward)

```python
import numpy as np
from scipy import stats

# Proyección con slope post-2016
ts_post_vals = ts_post.values
years_post = np.arange(len(ts_post_vals))
slope, intercept, r, p, se = stats.linregress(years_post, ts_post_vals)

# Proyectar 5 años hacia adelante
horizon = 5
future_years = np.arange(len(ts_post_vals), len(ts_post_vals) + horizon)
proyeccion = intercept + slope * future_years
print(f"Proyección al año {ts_post.index[-1].year + horizon}: {proyeccion[-1]:.1f}%")
print(f"(IC 95% aproximado: ± {1.96 * se * horizon:.1f} pp)")
```

### Paso 6: Consideraciones especiales para este análisis

- Si la serie tiene menos de 10 puntos por subperíodo, el Mann-Kendall no tendrá potencia suficiente. Reportar el slope con su intervalo de confianza y ser explícito sobre la limitación.
- Documentar cualquier cambio metodológico entre versiones del mapa de coberturas.
- Relacionar los resultados con el diagnóstico territorial del POT vigente del municipio.
- Registrar decisiones en [`docs/decisiones.md`](../../docs/decisiones.md).